# Get Patch-level label for Camelyon16 dataset
For the Cemlyon16 dataset, based on its dataset's annotation, we could get the pixel-level annotation. So, for each patch we could get the tumor, normal, and background proportion. We define the Tumor if the patch only has tumor region, Mix if there is partial tumor and partial normal. for normal tissue, we consider the rest as normal. 

## Analysis of patch-level label proportion

In [ ]:
import os
import sys
import pandas as pd

# put your model path here, e.g., if your dataset handler module is in ./models/xMIL/datasets/data_handler.py, then add ./models/xMIL to the path
sys.path.insert(0, "./models/xMIL")

# remove cached external package
if "datasets" in sys.modules:
    del sys.modules["datasets"]

from datasets.data_handler import SlideDataHandler

wsi_metadata_folder = "path/to/your/camelyon_annotated_metadata" # Update this to your Camelyon16-labeled patch metadata directory. Our dataset structure is /wsi_folde/slide_id/metadata/df.csv

rows = []

# iterate over each WSI folder
for wsi in os.listdir(wsi_metadata_folder):
    wsi_path = os.path.join(wsi_metadata_folder, wsi)

    if not os.path.isdir(wsi_path):
        continue

    csv_path = os.path.join(wsi_path, "metadata", "df.csv")

    if not os.path.exists(csv_path):
        print(f"Skip {wsi}: df.csv not found")
        continue

    df_csv = pd.read_csv(csv_path)

    # get patch labels
    df_csv["patch_label"] = SlideDataHandler.get_patch_labels(df_csv, "camelyon")

    # normalize to proportions
    proportions = df_csv["patch_label"].value_counts()

    rows.append({
        "slide_id": wsi,
        "tumor_proportion": proportions.get("Tumor", 0.0),
        "mix_proportion": proportions.get("Mix", 0.0),
        "normal_proportion": proportions.get("Normal", 0.0),
    })

df_proportion = pd.DataFrame(rows)

## Visualization of patch-type proportions

In [ ]:
import matplotlib.pyplot as plt

# -----------------------------
# Settings
# -----------------------------
SAVE_PATH = "/path/to/save/patch_type_proportion_three_panel.pdf"  # Update this

PATCH_LABEL_COL = "patch_labels"
SLIDE_COL = "slide_id"        # change if your slide id column has another name
REF_SLIDE_COL = "slide_id"    # change if df_reference uses another name
CLASS_COL = "class"

# -----------------------------
# Plot settings
# -----------------------------
ORDER = ["Tumor", "Mix", "Normal"]

plt.rcParams.update({
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "font.family": "serif",
    "font.serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset": "cm",
    "svg.fonttype": "none",  
})

# -----------------------------
# 1. Compute per-slide patch-type proportions
# -----------------------------
df_reference = pd.read_csv('/path/to/your/camelyon_reference.csv') # Update this to your reference CSV that contains slide-level labels (macro/micro/negative)


# -----------------------------
# 2. Match macro / micro slides
# -----------------------------
df_ref = df_reference.copy()

# if df_reference has image names like "tumor_001.tif", create slide_id
if REF_SLIDE_COL not in df_ref.columns and "image" in df_ref.columns:
    df_ref[REF_SLIDE_COL] = df_ref["image"].astype(str).str.split(".").str[0]
df_positive = pd.read_csv('/path/to/your/positive_slides_list.csv') # Update this to your CSV that contains the list of positive slides (slide_id column)
positive_ids = set(df_positive["slide_id"].astype(str))

df_ref = df_ref[df_ref[REF_SLIDE_COL].isin(positive_ids)].copy()
macro_ids = set(df_ref.loc[df_ref[CLASS_COL].str.lower() == "macro", REF_SLIDE_COL])
micro_ids = set(df_ref.loc[df_ref[CLASS_COL].str.lower() == "micro", REF_SLIDE_COL])

df_total = df_proportion.copy()
df_macro = df_proportion[df_proportion[SLIDE_COL].isin(macro_ids)].copy()
df_micro = df_proportion[df_proportion[SLIDE_COL].isin(micro_ids)].copy()

# -----------------------------
# 3. Create figure
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(6.8, 2.4), sharey=True)

panel_data = [
    ("All Camelyon16 Tumor slides", df_total),
    ("Macro-metastasis Tumor slides", df_macro),
    ("Micro-metastasis Tumor slides", df_micro),
]

for ax, (title, data) in zip(axes, panel_data):
    values = [data[col].dropna().values for col in ORDER]

    bp = ax.boxplot(
        values,
        labels=ORDER,
        patch_artist=True,
        widths=0.55,
        showfliers=False,
        medianprops=dict(color="#ff7f0e", linewidth=1.0),
        boxprops=dict(facecolor="#1f77b4", edgecolor="black", linewidth=0.8),
        whiskerprops=dict(color="black", linewidth=0.8),
        capprops=dict(color="black", linewidth=0.8),
    )

    ax.set_title(title, pad=4)
    ax.set_ylim(0, 1.0)

    # background like your example
    ax.set_facecolor("#EAEAF2")

    # horizontal dashed grid
    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.7)
    ax.grid(axis="x", visible=False)

    # spine style
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

axes[0].set_ylabel("Patch proportion")

plt.tight_layout(w_pad=1.2)
plt.savefig("patch_type_proportion_three_panel.svg",
            format="svg",
            bbox_inches="tight",
            pad_inches=0.02)
plt.show()

![figure](camelyon_patch_proportion.png)